In [1]:
import pandas as pd
import numpy as np
from ts_inverse.datahandler import convert_tsf_to_dataframe

/scratch/ejk5818/miniconda3/envs/ts-inverse/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:

def tsf_frequency_to_pandas_frequency(tsf_frequency):
    if tsf_frequency == 'daily':
        return 'D'
    elif tsf_frequency == 'hourly':
        return 'h'
    elif tsf_frequency == 'half_hourly':
        return '30min'
    return 'IDK'


def read_tsf_and_convert_to_csv(file_path, out_path):
    loaded_data, frequency, forecast_horizon, contain_missing_values, contain_equal_length = convert_tsf_to_dataframe(file_path)
    frequency = tsf_frequency_to_pandas_frequency(frequency)
    loaded_data_series_names = loaded_data['series_name'].values.tolist()
    min_timestamp = loaded_data['start_timestamp'].min()
    max_timestamp = loaded_data['start_timestamp'].max()

    # print(min_timestamp, max_timestamp)

    for name in loaded_data_series_names:
        loaded_data_series = loaded_data[loaded_data['series_name'] == name]
        series_values = loaded_data_series['series_value'].values[0]
        time_range = pd.date_range(start=loaded_data_series['start_timestamp'].values[0], periods=len(series_values), freq=frequency)
        if min(time_range) < min_timestamp:
            min_timestamp = min(time_range)

        if max(time_range) > max_timestamp:
            max_timestamp = max(time_range)

    # print(max_timestamp)

    columns = {}
    columns['Time'] = pd.date_range(start=min_timestamp, end=max_timestamp, freq=frequency)
    extra_identifiers = loaded_data.columns[:-2]
    for name in loaded_data_series_names:
        loaded_data_series = loaded_data[loaded_data['series_name'] == name]
        until_start_range = pd.date_range(start=min_timestamp, end=loaded_data_series['start_timestamp'].values[0], freq=frequency)

        if len(until_start_range) == 1:
            begin_padding = []
        else:
            begin_padding = [np.nan for _ in until_start_range][1:]
            
        series_values = loaded_data_series['series_value'].values[0].tolist()
        end_padding = [np.nan for _ in range(len(columns['Time']) - len(begin_padding) - len(series_values))]

        complete_name = f'{"_".join(loaded_data_series[extra_identifiers].values[0])}'
        columns[complete_name] = begin_padding + series_values + end_padding
        # print(complete_name, len(begin_padding), len(series_values), len(end_padding), len(columns[complete_name]))
    # print('Time', len(columns['Time']))

    df = pd.DataFrame(columns)
    df.set_index('Time', inplace=True)
    df.to_csv(out_path, index=True)
    return df, frequency, forecast_horizon, contain_missing_values, contain_equal_length

# Pre-process datasets and convert to CSV files

In [3]:
read_tsf_and_convert_to_csv("./KDDCup_2018/kdd_cup_2018_dataset_without_missing_values.tsf", "./KDDCup_2018/kdd_cup_2018_dataset_without_missing_values.csv")[0]

,T1_Beijing_aotizhongxin_aq_PM2.5,T2_Beijing_aotizhongxin_aq_PM10,T3_Beijing_aotizhongxin_aq_NO2,T4_Beijing_aotizhongxin_aq_CO,T5_Beijing_aotizhongxin_aq_O3,T6_Beijing_aotizhongxin_aq_SO2,T7_Beijing_badaling_aq_PM2.5,T8_Beijing_badaling_aq_PM10,T9_Beijing_badaling_aq_NO2,T10_Beijing_badaling_aq_CO,...,T261_London_RB7_PM10,T262_London_RB7_NO2,T263_London_BX1_PM2.5,T264_London_BX1_PM10,T265_London_BX1_NO2,T266_London_BX9_PM2.5,T267_London_KC1_PM2.5,T268_London_KC1_PM10,T269_London_KC1_NO2,T270_London_CT2_PM2.5
Time,,,,,,,,,,,,,,,,,,,,,
2017-01-01 00:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,55.0,22.0,23.3,16.1,7.0,23.3,31.7,23.2,21.3,NaN
2017-01-01 01:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,26.0,27.0,17.5,10.7,9.4,17.5,29.9,23.1,25.9,NaN
2017-01-01 02:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,24.4,22.0,16.6,8.9,7.8,16.6,21.8,13.9,22.6,NaN
2017-01-01 03:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,15.8,18.6,18.6,8.5,5.4,18.6,19.1,12.8,23.4,NaN
2017-01-01 04:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22.4,16.3,20.4,8.3,5.4,20.4,23.7,13.8,22.6,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-03-31 19:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,9.8,23.6,10.2,5.9,1.4,4.5,44.8,18.7,16.9,6.0
2018-03-31 20:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,9.8,23.6,10.2,5.9,1.4,4.5,44.8,18.7,16.9,6.0
2018-03-31 21:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,9.8,23.6,10.2,5.9,1.4,4.5,44.8,18.7,16.9,6.0


In [4]:
read_tsf_and_convert_to_csv("./LondonSmartMeter/london_smart_meters_dataset_without_missing_values.tsf", "./LondonSmartMeter/london_smart_meters_dataset_without_missing_values.csv")[0]

,T1,T2,T3,T4,T5,T6,T7,T8,T9,T10,...,T5551,T5552,T5553,T5554,T5555,T5556,T5557,T5558,T5559,T5560
Time,,,,,,,,,,,,,,,,,,,,,
2011-11-24 00:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-11-24 00:30:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-11-24 01:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-11-24 01:30:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-11-24 02:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-02-27 21:30:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.766,0.007,NaN,NaN,NaN,NaN
2014-02-27 22:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.748,0.006,NaN,NaN,NaN,NaN
2014-02-27 22:30:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.792,0.005,NaN,NaN,NaN,NaN


## Create subset of londom smartmeter and Electricity 370 datasets

In [5]:
df_london_smartmeter = pd.read_csv("./LondonSmartMeter/london_smart_meters_dataset_without_missing_values.csv")
#Make subselection of the first 30 columns
df_london_smartmeter = df_london_smartmeter.iloc[:, 0:30]
df_london_smartmeter.set_index('Time', inplace=True)
df_london_smartmeter.to_csv("./LondonSmartMeter/london_smart_meters_dataset_without_missing_values_first_30_consumers.csv")

In [6]:
df_electricity_370 = pd.read_csv("./Electricity370/LD2011_2014.txt", delimiter=';', decimal=',')
# Give column 0 the name Time
df_electricity_370.rename(columns={df_electricity_370.columns[0]: 'Time'}, inplace=True)
# Convert the Time column to datetime and make it the index
df_electricity_370['Time'] = pd.to_datetime(df_electricity_370['Time'])
df_electricity_370.set_index('Time', inplace=True)
df_electricity_370.head()
df_electricity_370.to_csv("./Electricity370/LD2011_2014.csv")

df_electricity_370_first_40_columns = df_electricity_370.iloc[:, 0:40]
df_electricity_370_first_40_columns.to_csv("./Electricity370/LD2011_2014_first_40_consumers.csv")

In [7]:
read_tsf_and_convert_to_csv("./Electricity321Hourly/electricity_hourly_dataset.tsf", "./Electricity321Hourly/electricity_hourly_dataset.csv")[0]

,T1,T2,T3,T4,T5,T6,T7,T8,T9,T10,...,T312,T313,T314,T315,T316,T317,T318,T319,T320,T321
Time,,,,,,,,,,,,,,,,,,,,,
2012-01-01 00:00:01,14.0,69.0,234.0,415.0,215.0,1056.0,29.0,840.0,226.0,265.0,...,676.0,372.0,80100.0,4719.0,5002.0,48.0,38.0,1558.0,182.0,2162.0
2012-01-01 01:00:01,18.0,92.0,312.0,556.0,292.0,1363.0,29.0,1102.0,271.0,340.0,...,805.0,452.0,95200.0,4643.0,6617.0,65.0,47.0,2177.0,253.0,2835.0
2012-01-01 02:00:01,21.0,96.0,312.0,560.0,272.0,1240.0,29.0,1025.0,270.0,300.0,...,817.0,430.0,96600.0,4285.0,6571.0,64.0,43.0,2193.0,218.0,2764.0
2012-01-01 03:00:01,20.0,92.0,312.0,443.0,213.0,845.0,24.0,833.0,179.0,211.0,...,801.0,291.0,94500.0,4222.0,6365.0,65.0,39.0,1315.0,195.0,2735.0
2012-01-01 04:00:01,22.0,91.0,312.0,346.0,190.0,647.0,16.0,733.0,186.0,179.0,...,807.0,279.0,91300.0,4116.0,6298.0,75.0,40.0,1378.0,191.0,2721.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-12-31 19:00:01,11.0,116.0,8.0,844.0,384.0,1590.0,51.0,1412.0,407.0,351.0,...,1897.0,1589.0,166500.0,9917.0,10412.0,324.0,21.0,1870.0,162.0,2773.0
2014-12-31 20:00:01,11.0,103.0,8.0,749.0,371.0,1366.0,47.0,1265.0,369.0,327.0,...,1374.0,1336.0,158800.0,6812.0,8956.0,302.0,20.0,1506.0,438.0,2755.0
2014-12-31 21:00:01,12.0,93.0,8.0,650.0,346.0,1282.0,48.0,1079.0,308.0,283.0,...,938.0,1311.0,154300.0,6602.0,5910.0,302.0,18.0,1864.0,621.0,2650.0
